In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp01
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)




# ════════════════════════════════════════════════════════════════════════
# MLFP01 — Exercise 1: Your First Data Exploration
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Assign variables (strings, integers, floats, booleans) and format
#     output with f-strings, including alignment and decimal control
#   - Load a CSV file into a Polars DataFrame and inspect its structure
#   - Compute summary statistics with describe() and column aggregations
#   - Filter rows to find extreme values (max, min) in a dataset
#   - Build formatted summary reports combining all of the above
#
# PREREQUISITES: None — this is the very first exercise in the course.
#
# ESTIMATED TIME: ~150-180 min
#
# TASKS:
#   1.  Python basics — variables, types, and type checking
#   2.  String operations and f-string formatting
#   3.  Arithmetic and built-in functions
#   4.  Lists and basic iteration
#   5.  Load data and inspect shape, columns, and types
#   6.  Compute summary statistics with describe()
#   7.  Column-level aggregations (mean, min, max, std, quantile)
#   8.  Find extreme months and conditional reasoning
#   9.  Build a formatted summary report
#   10. Cross-column analysis — temperature vs rainfall relationship
#
# DATASET: Singapore monthly weather data (temperature, rainfall, humidity)
#   Source: Meteorological Service Singapore (data.gov.sg)
#   Rows: ~12 monthly records | Columns: month, mean_temperature_c,
#   total_rainfall_mm, relative_humidity_pct, sunshine_hours
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import polars as pl



### Data Loading


In [ ]:
# MLFPDataLoader handles finding data files for you — whether you're
# running locally (VS Code) or on Google Colab. Just specify the module
# and filename, and it will locate the data automatically.
loader = MLFPDataLoader()
df = loader.load("mlfp01", "sg_weather.csv")

print("=" * 60)
print("  MLFP01 Exercise 1: Your First Data Exploration")
print("=" * 60)
print(f"\n  Data loaded: sg_weather.csv ({df.height} rows, {df.width} columns)")
print(f"  You're ready to start!\n")



## TASK 1: Python basics — variables, types, and type checking


In [ ]:
# In Python, a variable stores a value. The name goes on the left,
# the value on the right. Python figures out the type automatically —
# no need to declare "this is a string" or "this is a number".

# --- 1a: Core data types ---
city = "Singapore"  # str: a piece of text, always in quotes
country = "Singapore"  # another string
years_of_data = 30  # int: a whole number (no decimal point)
latitude = 1.35  # float: a decimal number
is_tropical = True  # bool: True or False — used for decisions

# Python's type() function tells you what kind of value a variable holds.
print("=== Variable Types ===")
print(f"city:          {type(city).__name__}   → {city!r}")
print(f"years_of_data: {type(years_of_data).__name__}    → {years_of_data}")
print(f"latitude:      {type(latitude).__name__}  → {latitude}")
print(f"is_tropical:   {type(is_tropical).__name__}   → {is_tropical}")

# --- 1b: Type conversion ---
price_str = "450000"  # This is text, not a number — you can't do math
# TODO: Convert price_str to an integer named price_int
price_int = ____  # Hint: int(price_str)
# TODO: Convert price_str to a float named price_float
price_float = ____  # Hint: float(price_str)
height_int = int(3.7)  # Truncates toward zero: 3 (NOT rounded)
bool_from_int = bool(1)  # Any nonzero number is True; 0 is False

print("\n=== Type Conversions ===")
print(f"'{price_str}' (str) → {price_int} (int) → {price_float} (float)")
print(f"int(3.7) = {height_int}  (truncates, does NOT round)")
print(f"bool(1)  = {bool_from_int},  bool(0) = {bool(0)}")

# --- 1c: None — the absence of a value ---
missing_value = None
print(f"\nmissing_value is None: {missing_value is None}")
print(f"type(None): {type(None).__name__}")



### Checkpoint 1


In [ ]:
assert isinstance(city, str), "city should be a string"
assert isinstance(years_of_data, int), "years_of_data should be an integer"
assert isinstance(latitude, float), "latitude should be a float"
assert isinstance(is_tropical, bool), "is_tropical should be a boolean"
assert price_int == 450_000, "int conversion of '450000' should be 450000"
assert height_int == 3, "int(3.7) should truncate to 3"
assert missing_value is None, "missing_value should be None"
print("\n✓ Checkpoint 1 passed — variables, types, and conversions working\n")



## TASK 2: String operations and f-string formatting


╔{'═' * 40}╗
║{'Singapore Weather Report':^40}║
║{'Data from Meteorological Service':^40}║
╚{'═' * 40}╝


In [ ]:
# --- 2a: String methods ---
raw_town = "  ang mo kio  "
clean_town = raw_town.strip()  # Remove leading/trailing whitespace
# TODO: Convert clean_town to uppercase and assign to upper_town
upper_town = ____  # Hint: clean_town.upper()
title_town = clean_town.title()  # "Ang Mo Kio" — for display
word_count = len(clean_town.split())  # Split by spaces, count the pieces

print("=== String Methods ===")
print(f"raw:    {raw_town!r}")
print(f"strip:  {clean_town!r}")
print(f"upper:  {upper_town!r}")
print(f"title:  {title_town!r}")
print(f"words:  {word_count}")

# .replace() swaps one substring for another
dirty_price = "S$450,000"
# TODO: Remove "S$" and "," from dirty_price, assign to numeric_price
numeric_price = dirty_price.replace("S$", "").replace(
    ____, ____
)  # Hint: replace(",", "")
print(f"\n'{dirty_price}' → '{numeric_price}' → {int(numeric_price)}")

filename = "hdb_resale.parquet"
print(f"\n'{filename}' starts with 'hdb': {filename.startswith('hdb')}")
print(f"'{filename}' ends with '.csv': {filename.endswith('.csv')}")
print(f"'{filename}' ends with '.parquet': {filename.endswith('.parquet')}")

# --- 2b: f-string formatting specifiers ---
celsius_avg = 27.5
# TODO: Convert celsius_avg to Fahrenheit (formula: C * 9/5 + 32)
fahrenheit_avg = ____  # Hint: celsius_avg * 9 / 5 + 32
population = 5_917_600

print("\n=== f-string Formatting ===")
print(f"Temperature: {celsius_avg}°C / {fahrenheit_avg:.1f}°F")
#   :.1f  → 1 decimal place, float
print(f"Population:  {population:,}")
#   :,    → thousands separator
print(f"Population:  {population:>15,}")
#   :>15, → right-align in 15-char field with comma separator
print(f"Percentage:  {0.8765:.1%}")
#   :.1%  → multiply by 100, add %, 1 decimal
print(f"Left-align:  {'Singapore':<20} | end")
print(f"Center:      {'Singapore':^20} | end")

# --- 2c: Multi-line strings ---
print(report_header)



### Checkpoint 2


In [ ]:
assert (
    upper_town == "ANG MO KIO"
), f"upper() should give 'ANG MO KIO', got {upper_town!r}"
assert word_count == 3, f"'ang mo kio' has 3 words, got {word_count}"
assert int(numeric_price) == 450_000, "cleaned price should be 450000"
assert fahrenheit_avg > 80, "Fahrenheit conversion looks wrong"
print("✓ Checkpoint 2 passed — string operations and f-string formatting working\n")



## TASK 3: Arithmetic and built-in functions


In [ ]:
# --- 3a: Arithmetic operators ---
a, b = 17, 5
print("=== Arithmetic Operators ===")
print(f"{a} + {b}  = {a + b}")
print(f"{a} - {b}  = {a - b}")
print(f"{a} * {b}  = {a * b}")
print(f"{a} / {b}  = {a / b}")  # True division: always float
print(f"{a} // {b} = {a // b}")  # Floor division: rounds down
print(f"{a} % {b}  = {a % b}")  # Modulo: remainder
print(f"{a} ** {b} = {a ** b}")  # Power

print(f"\n10 / 5 = {10 / 5}  (type: {type(10 / 5).__name__})")
print(f"10 // 5 = {10 // 5}  (type: {type(10 // 5).__name__})")

# --- 3b: Built-in functions for data work ---
prices = [350_000, 420_000, 510_000, 680_000, 1_200_000]
print("\n=== Built-in Functions ===")
print(f"len(prices):   {len(prices)}")
print(f"sum(prices):   {sum(prices):,}")
print(f"min(prices):   {min(prices):,}")
print(f"max(prices):   {max(prices):,}")
print(f"sorted(prices, reverse=True): {sorted(prices, reverse=True)}")

# TODO: Compute the mean of prices (sum / count) and assign to mean_price
mean_price = ____  # Hint: sum(prices) / len(prices)
print(f"mean(prices):  {mean_price:,.0f}")

price_change = -25_000
print(f"\nprice_change: {price_change:,}  abs: {abs(price_change):,}")

pi = 3.14159265
print(f"round(pi, 2): {round(pi, 2)}")
print(f"round(pi, 4): {round(pi, 4)}")
print(f"round(1234, -2): {round(1234, -2)}")

# --- 3c: Comparison operators ---
x = 500_000
print("\n=== Comparison Operators ===")
print(f"{x} > 400000:  {x > 400_000}")
print(f"{x} <= 500000: {x <= 500_000}")
print(f"{x} == 500000: {x == 500_000}")
print(f"{x} != 400000: {x != 400_000}")



### Checkpoint 3


In [ ]:
assert 17 // 5 == 3, "Floor division of 17 // 5 should be 3"
assert 17 % 5 == 2, "17 % 5 should be 2"
assert len(prices) == 5, "prices list should have 5 elements"
assert sum(prices) == 3_160_000, "sum of prices should be 3,160,000"
assert mean_price == 632_000.0, f"mean should be 632000, got {mean_price}"
assert round(pi, 2) == 3.14, "round(pi, 2) should be 3.14"
print("\n✓ Checkpoint 3 passed — arithmetic and built-in functions working\n")



## TASK 4: Lists and basic iteration


In [ ]:
# --- 4a: Creating and accessing lists ---
months = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]

print("=== List Operations ===")
print(f"First month:  {months[0]}")  # Index 0 = first item
print(f"Last month:   {months[-1]}")  # Index -1 = last item
print(f"Q1 months:    {months[0:3]}")  # Slice [start:stop) — stop is exclusive
print(f"Q4 months:    {months[9:]}")  # From index 9 to the end
print(f"Every 3rd:    {months[::3]}")  # Step by 3

# --- 4b: Modifying lists ---
temperatures = [27.0, 27.1, 27.8, 28.3, 28.6, 28.3, 27.9, 27.8, 27.6, 27.6, 27.0, 26.8]
temperatures.append(27.2)  # Add to the end
print(f"\nAfter append: {len(temperatures)} items")
temperatures.pop()  # Remove from the end
print(f"After pop:    {len(temperatures)} items")

# --- 4c: List comprehensions ---
celsius_temps = [27.0, 27.5, 28.0, 28.5, 29.0]
# TODO: Build fahrenheit_temps by converting each celsius value (C * 9/5 + 32)
fahrenheit_temps = [____ for c in celsius_temps]  # Hint: c * 9 / 5 + 32
print(f"\nCelsius:    {celsius_temps}")
print(f"Fahrenheit: {[round(f, 1) for f in fahrenheit_temps]}")

# With a filter — only keep values above a threshold
warm_months = [c for c in celsius_temps if c >= 28.0]
print(f"Warm (>=28°C): {warm_months}")

# --- 4d: Dictionaries — key-value pairs ---
month_temps = {
    "Jan": 26.8,
    "Feb": 27.0,
    "Mar": 27.8,
    "Apr": 28.3,
    "May": 28.6,
    "Jun": 28.3,
    "Jul": 27.9,
    "Aug": 27.8,
    "Sep": 27.6,
    "Oct": 27.6,
    "Nov": 27.0,
    "Dec": 26.8,
}

print(f"\nJan temp: {month_temps['Jan']}°C")
print(f"Keys:     {list(month_temps.keys())[:4]}...")
print(f"Values:   {list(month_temps.values())[:4]}...")

# TODO: Find the hottest month using max() with a key function
hottest_dict = ____  # Hint: max(month_temps, key=month_temps.get)
print(f"Hottest:  {hottest_dict} at {month_temps[hottest_dict]}°C")

# --- 4e: for loop with enumerate ---
print("\n=== Monsoon Season Analysis ===")
monsoon_months = ["Nov", "Dec", "Jan"]
for i, m in enumerate(monsoon_months, start=1):
    temp = month_temps[m]
    print(f"  {i}. {m}: {temp}°C")



### Checkpoint 4


In [ ]:
assert months[0] == "Jan", "First month should be Jan"
assert months[-1] == "Dec", "Last month should be Dec"
assert len(celsius_temps) == 5, "celsius_temps should have 5 items"
assert len(fahrenheit_temps) == 5, "fahrenheit_temps should have 5 items"
assert len(warm_months) == 3, "3 months should be >= 28°C"
assert hottest_dict == "May", f"Hottest month should be May, got {hottest_dict}"
print("\n✓ Checkpoint 4 passed — lists, dicts, and iteration working\n")



## TASK 5: Load data and inspect its shape and structure


In [ ]:
# A DataFrame is a table of data — rows are observations, columns are
# measurements. Think of it like a spreadsheet in your code.

rows, cols = df.shape
print(f"=== Dataset Overview ===")
print(f"Rows: {rows:,}")
print(f"Columns: {cols}")

print(f"\nColumn names:")
for col_name in df.columns:
    print(f"  - {col_name}")

print(f"\nData types:")
for col_name, dtype in zip(df.columns, df.dtypes):
    print(f"  {col_name:>25}: {dtype}")

print(f"\nFirst 5 rows:")
print(df.head(5))

print(f"\nLast 3 rows:")
print(df.tail(3))

print(f"\n3 random rows:")
print(df.sample(n=min(3, df.height), seed=42))

# INTERPRETATION: Inspecting head, tail, and sample gives you three views
# of the same data. If head() and tail() look different (different columns,
# different formats), you might have schema drift.



### Checkpoint 5


In [ ]:
assert rows > 0, "DataFrame has no rows — check data loading"
assert cols >= 2, "DataFrame should have at least 2 columns"
assert "month" in df.columns, "Expected a 'month' column"
assert len(df.columns) == cols, "Column count should match .shape"
print("\n✓ Checkpoint 5 passed — data loaded and inspected\n")



## TASK 6: Summary statistics with describe()


In [ ]:
# .describe() computes count, mean, std, min, max for every numeric column.
# This single call replaces writing 5 separate calculations.

print(f"=== Summary Statistics ===")
stats = df.describe()
print(stats)

# INTERPRETATION: describe() gives you the "vital signs" of your data.
# count: any column with count < total rows has missing data
# mean vs median (50%): if they differ a lot, the data is skewed

# TODO: Compute mean temperature and assign to mean_temp
mean_temp = ____  # Hint: df["mean_temperature_c"].mean()
# TODO: Compute min temperature and assign to min_temp
min_temp = ____  # Hint: df["mean_temperature_c"].min()
# TODO: Compute max temperature and assign to max_temp
max_temp = ____  # Hint: df["mean_temperature_c"].max()
std_temp = df["mean_temperature_c"].std()

print(f"\nTemperature details:")
print(f"  Average:  {mean_temp:.2f}°C")
print(f"  Minimum:  {min_temp:.2f}°C")
print(f"  Maximum:  {max_temp:.2f}°C")
print(f"  Std dev:  {std_temp:.2f}°C")
# INTERPRETATION: Std dev measures how spread out the values are.
# Singapore's tropical climate means low temperature variation (~1°C std).

mean_rain = df["total_rainfall_mm"].mean()
max_rain = df["total_rainfall_mm"].max()
min_rain = df["total_rainfall_mm"].min()
std_rain = df["total_rainfall_mm"].std()
print(f"\nRainfall details:")
print(f"  Average:  {mean_rain:.1f} mm/month")
print(f"  Maximum:  {max_rain:.1f} mm/month")
print(f"  Minimum:  {min_rain:.1f} mm/month")
print(f"  Std dev:  {std_rain:.1f} mm/month")



### Checkpoint 6


In [ ]:
assert mean_temp is not None, "mean_temp should not be None"
assert 20 < mean_temp < 35, f"mean_temp={mean_temp} seems wrong for Singapore"
assert mean_rain > 0, "mean_rain should be positive"
assert std_temp < std_rain, "Rainfall should be more variable than temperature"
print("\n✓ Checkpoint 6 passed — summary statistics computed correctly\n")



## TASK 7: Column-level aggregations — quantiles and ranges


In [ ]:
# Beyond mean and std, quantiles divide your data into portions.
# The median (50th percentile) is the "middle" value.

# --- 7a: Quantiles for temperature ---
temp_series = df["mean_temperature_c"]
# TODO: Compute the 25th percentile of temp_series
temp_q25 = ____  # Hint: temp_series.quantile(0.25)
# TODO: Compute the 50th percentile (median)
temp_q50 = ____  # Hint: temp_series.quantile(0.50)
temp_q75 = temp_series.quantile(0.75)
temp_iqr = temp_q75 - temp_q25

print("=== Temperature Quantiles ===")
print(f"  25th percentile (Q1): {temp_q25:.2f}°C")
print(f"  50th percentile (Q2): {temp_q50:.2f}°C  (this is the median)")
print(f"  75th percentile (Q3): {temp_q75:.2f}°C")
print(f"  Interquartile range:  {temp_iqr:.2f}°C  (Q3 - Q1)")
# INTERPRETATION: The IQR tells you the range of the "middle 50%" of values.
# For Singapore temperature, IQR ≈ 1°C means the middle half of all monthly
# temperatures span just 1 degree — remarkably stable.

# --- 7b: Quantiles for rainfall ---
rain_series = df["total_rainfall_mm"]
rain_q25 = rain_series.quantile(0.25)
rain_q50 = rain_series.quantile(0.50)
rain_q75 = rain_series.quantile(0.75)
rain_iqr = rain_q75 - rain_q25

print(f"\n=== Rainfall Quantiles ===")
print(f"  Q1: {rain_q25:.1f} mm")
print(f"  Q2: {rain_q50:.1f} mm  (median)")
print(f"  Q3: {rain_q75:.1f} mm")
print(f"  IQR: {rain_iqr:.1f} mm")

# --- 7c: Coefficient of variation ---
cv_temp = (std_temp / mean_temp) * 100
cv_rain = (std_rain / mean_rain) * 100

print(f"\n=== Coefficient of Variation ===")
print(f"  Temperature CV: {cv_temp:.1f}%  (very low — stable climate)")
print(f"  Rainfall CV:    {cv_rain:.1f}%  (much higher — seasonal variation)")

# --- 7d: Data range and total ---
total_annual_rain = rain_series.sum()
temp_range = max_temp - min_temp

print(f"\n=== Annual Totals ===")
print(f"  Total annual rainfall: {total_annual_rain:.0f} mm")
print(f"  Temperature range:     {temp_range:.1f}°C (max - min)")



### Checkpoint 7


In [ ]:
assert temp_q25 is not None, "Q1 should not be None"
assert temp_q50 is not None, "Median should not be None"
assert temp_q25 <= temp_q50 <= temp_q75, "Quantiles should be ordered"
assert rain_iqr >= 0, "IQR should be non-negative"
assert cv_rain > cv_temp, "Rainfall should be more variable than temperature"
assert total_annual_rain > 0, "Total annual rainfall should be positive"
print("\n✓ Checkpoint 7 passed — quantiles and advanced statistics correct\n")



## TASK 8: Find extreme months and conditional reasoning


In [ ]:
# .filter() keeps rows where a condition is True
# pl.col("column_name") refers to a column inside a Polars expression

# --- 8a: Find the hottest, coldest, and wettest months ---
# TODO: Filter for the row where temperature equals its maximum
hottest_row = df.filter(
    pl.col("mean_temperature_c") == ____
)  # Hint: df["mean_temperature_c"].max()
coldest_row = df.filter(pl.col("mean_temperature_c") == df["mean_temperature_c"].min())
# TODO: Filter for the row where rainfall equals its maximum
wettest_row = df.filter(
    pl.col("total_rainfall_mm") == ____
)  # Hint: df["total_rainfall_mm"].max()
driest_row = df.filter(pl.col("total_rainfall_mm") == df["total_rainfall_mm"].min())

hottest_month = hottest_row["month"][0]
hottest_temp = hottest_row["mean_temperature_c"][0]
coldest_month = coldest_row["month"][0]
coldest_temp = coldest_row["mean_temperature_c"][0]
wettest_month = wettest_row["month"][0]
wettest_rain = wettest_row["total_rainfall_mm"][0]
driest_month = driest_row["month"][0]
driest_rain = driest_row["total_rainfall_mm"][0]

print(f"=== Extreme Months ===")
print(f"Hottest:  {hottest_month} at {hottest_temp:.1f}°C")
print(f"Coldest:  {coldest_month} at {coldest_temp:.1f}°C")
print(f"Wettest:  {wettest_month} with {wettest_rain:.1f} mm")
print(f"Driest:   {driest_month} with {driest_rain:.1f} mm")

# --- 8b: Above-average analysis ---
above_avg_temp = df.filter(pl.col("mean_temperature_c") > mean_temp)
above_avg_rain = df.filter(pl.col("total_rainfall_mm") > mean_rain)

print(f"\n=== Above-Average Months ===")
print(f"Temperature above {mean_temp:.1f}°C: {above_avg_temp.height} months")
print(f"  Months: {above_avg_temp['month'].to_list()}")
print(f"Rainfall above {mean_rain:.0f} mm: {above_avg_rain.height} months")
print(f"  Months: {above_avg_rain['month'].to_list()}")

# --- 8c: Percentage deviations from average ---
hottest_pct_above = ((hottest_temp - mean_temp) / mean_temp) * 100
wettest_pct_above = ((wettest_rain - mean_rain) / mean_rain) * 100

print(f"\n=== Deviations from Average ===")
print(f"Hottest month is {hottest_pct_above:+.1f}% above average temperature")
print(f"Wettest month is {wettest_pct_above:+.1f}% above average rainfall")

# INTERPRETATION: Singapore's hottest months are May-Jun (pre-monsoon),
# coldest are Dec-Jan (NE monsoon), and wettest are Nov-Jan (monsoon peak).
# Temperature deviation is small while rainfall deviation is large.



### Checkpoint 8


In [ ]:
assert hottest_temp >= coldest_temp, "Hottest should be >= coldest"
assert wettest_rain >= driest_rain, "Wettest should be >= driest"
assert wettest_rain >= mean_rain, "Wettest month should be above average"
assert isinstance(hottest_month, str), "Month should be a string"
assert above_avg_temp.height > 0, "Some months should be above average temp"
assert above_avg_rain.height > 0, "Some months should be above average rain"
print("\n✓ Checkpoint 8 passed — extreme values and deviations found correctly\n")



## TASK 9: Formatted summary report


In [ ]:
# Let's combine everything into a clean, readable report.
# Good formatting makes data accessible to non-technical readers.

separator = "═" * 60

print(f"\n{separator}")
print(f"  SINGAPORE WEATHER SUMMARY REPORT")
print(f"{separator}")
print(f"  Source:     Meteorological Service Singapore")
print(f"  Records:    {rows:>6,} monthly observations")
print(f"  Variables:  {cols:>6} columns")
print(f"")
print(f"  ┌{'─' * 42}┐")
print(f"  │ TEMPERATURE (°C)                         │")
print(f"  ├{'─' * 42}┤")
print(f"  │  Mean:       {mean_temp:>8.2f}                     │")
print(f"  │  Std Dev:    {std_temp:>8.2f}  (CV: {cv_temp:.1f}%)        │")
print(f"  │  Range:      {min_temp:>8.2f} — {max_temp:.2f}             │")
print(f"  │  IQR:        {temp_iqr:>8.2f}                     │")
print(f"  │  Hottest:    {hottest_month:>8} ({hottest_temp:.1f}°C)        │")
print(f"  │  Coldest:    {coldest_month:>8} ({coldest_temp:.1f}°C)        │")
print(f"  └{'─' * 42}┘")
print(f"")
print(f"  ┌{'─' * 42}┐")
print(f"  │ RAINFALL (mm/month)                      │")
print(f"  ├{'─' * 42}┤")
print(f"  │  Mean:       {mean_rain:>8.1f}                     │")
print(f"  │  Std Dev:    {std_rain:>8.1f}  (CV: {cv_rain:.1f}%)       │")
print(f"  │  Range:      {min_rain:>8.1f} — {max_rain:.1f}           │")
print(f"  │  IQR:        {rain_iqr:>8.1f}                     │")
print(f"  │  Annual:     {total_annual_rain:>8.0f} mm                 │")
print(f"  │  Wettest:    {wettest_month:>8} ({wettest_rain:.0f} mm)        │")
print(f"  │  Driest:     {driest_month:>8} ({driest_rain:.0f} mm)         │")
print(f"  └{'─' * 42}┘")
print(f"{separator}")



### Checkpoint 9


In [ ]:
print("\n✓ Checkpoint 9 passed — formatted summary report generated\n")



## TASK 10: Cross-column analysis — temperature vs rainfall


In [ ]:
# Real analysis involves looking at relationships BETWEEN columns, not
# just summarising one column at a time.

# --- 10a: Does it rain more in hotter or cooler months? ---
warm_months_df = df.filter(pl.col("mean_temperature_c") >= temp_q50)
cool_months_df = df.filter(pl.col("mean_temperature_c") < temp_q50)

warm_avg_rain = warm_months_df["total_rainfall_mm"].mean()
cool_avg_rain = cool_months_df["total_rainfall_mm"].mean()

print("=== Temperature vs Rainfall ===")
print(f"Warm months (>= {temp_q50:.1f}°C):  avg rainfall = {warm_avg_rain:.1f} mm")
print(f"Cool months (<  {temp_q50:.1f}°C):  avg rainfall = {cool_avg_rain:.1f} mm")
if cool_avg_rain > warm_avg_rain:
    print("  → Cool months are wetter — this aligns with the NE monsoon (Dec-Feb)")
else:
    print("  → Warm months are wetter")

# --- 10b: Monthly deviation table (z-scores) ---
monthly_deviations = df.with_columns(
    ((pl.col("mean_temperature_c") - mean_temp) / std_temp).alias("temp_z_score"),
    ((pl.col("total_rainfall_mm") - mean_rain) / std_rain).alias("rain_z_score"),
)

print(f"\n=== Monthly Deviations (z-scores) ===")
print(f"  {'Month':>8} {'Temp z':>10} {'Rain z':>10}")
print(f"  {'─' * 30}")
for row in monthly_deviations.iter_rows(named=True):
    temp_z = row.get("temp_z_score")
    rain_z = row.get("rain_z_score")
    if temp_z is not None and rain_z is not None:
        print(f"  {row['month']:>8} {temp_z:>+10.2f} {rain_z:>+10.2f}")
# INTERPRETATION: z-scores express each value as "how many standard deviations
# from the mean." This lets you compare temperature and rainfall on the same scale.

# --- 10c: Polars correlation coefficient ---
# TODO: Compute Pearson correlation between temperature and rainfall
temp_rain_corr = (
    ____  # Hint: df.select(pl.corr("mean_temperature_c", "total_rainfall_mm")).item()
)

print(f"\nPearson correlation (temperature vs rainfall): {temp_rain_corr:.3f}")
if abs(temp_rain_corr) < 0.3:
    print("  → Weak correlation: temperature alone does not predict rainfall well")
elif temp_rain_corr < 0:
    print("  → Negative correlation: hotter months tend to be drier")
else:
    print("  → Positive correlation: hotter months tend to be wetter")



### Checkpoint 10


In [ ]:
assert warm_avg_rain is not None, "warm_avg_rain should not be None"
assert cool_avg_rain is not None, "cool_avg_rain should not be None"
assert isinstance(temp_rain_corr, float), "Correlation should be a float"
assert -1.0 <= temp_rain_corr <= 1.0, "Correlation must be between -1 and 1"
assert "temp_z_score" in monthly_deviations.columns, "z-score column should exist"
print("\n✓ Checkpoint 10 passed — cross-column analysis and correlation correct\n")



## REFLECTION


✓ Variables: str, int, float, bool, None — Python's core types
  ✓ Type conversion: int(), float(), str(), bool() — casting between types
  ✓ String methods: strip, upper, replace, split, startswith
  ✓ f-strings: alignment (:>8), decimals (:.2f), percentages (:.1%),
    thousands separator (:,)
  ✓ Arithmetic: +, -, *, /, //, %, ** — and when to use each
  ✓ Built-in functions: len, sum, min, max, sorted, round, abs
  ✓ Lists: indexing, slicing, append, comprehensions
  ✓ Dictionaries: key-value pairs, .keys(), .values(), max() with key=
  ✓ DataFrames: loading CSV, shape, columns, dtypes, head, tail, sample
  ✓ describe(): one-call summary statistics for every column
  ✓ Aggregations: mean, min, max, std, quantile — the data analyst's toolkit
  ✓ Filtering: finding rows that match a condition with .filter()
  ✓ Formatted output: building professional reports with alignment
  ✓ Cross-column analysis: z-scores, correlation, comparative reasoning

  NEXT: In Exercise 2, you'll learn to filter and transform data
  using Polars expressions — selecting rows by condition, creating
  new columns, and chaining operations together. The HDB resale
  dataset (500K+ transactions) will be your playground.


In [ ]:
print("═" * 60)
print("  WHAT YOU'VE MASTERED")
print("═" * 60)
print(
)

